# Chatterbox Multilingual V3: Custom-Text TTS Web UI

Run **Chatterbox Multilingual V3** in Google Colab and use it through a browser-based [Gradio](https://gradio.app/) interface. Enter your own text, choose a supported language, choose a built-in reference voice or upload/record your own voice reference, adjust generation settings, and generate downloadable WAV audio.

This is a **standalone TTS playground**. It does not read, modify, approve, or synthesize the *Buddhist Yoga* manuscript. It does not change the project's full-book synthesis gate.

## What it provides

- Free-form text input, with conservative sentence-aware chunking for longer passages.
- The language list obtained directly from the loaded Chatterbox V3 model.
- Built-in English and Hindi demo references, or your own uploaded/microphone-recorded reference audio.
- Controls for exaggeration, CFG weight, temperature, repetition penalty, min-p, top-p, and seed.
- WAV playback and download. Files are saved in the Colab session by default; Google Drive persistence is optional.
- The same pinned Chatterbox source revision and verified V3 model-file approach used by the project's existing notebook.

**Runtime:** GPU is strongly recommended. CPU fallback is supported but can be very slow. The selected model files require several gigabytes of download/storage.

**Google Drive is optional:** by default, the notebook does not mount or access Drive. The model and generated files stay in temporary Colab storage, and generated WAVs can be downloaded from Gradio. To keep model snapshots and generated WAVs/metadata across runtime resets, set `USE_GOOGLE_DRIVE = True` in the setup cell and rerun from the top. The first Drive-enabled run will ask you to authorize access.

**Privacy:** the Gradio share URL is reachable over the public internet, so this notebook requires a password for the UI. Keep both the URL and password private. Your uploaded voice reference is processed in the Colab runtime and is not copied into the repository or the Drive output folder. A normalized temporary copy remains under the Colab VM's `/content` directory until that runtime is terminated or reset. Generated JSON metadata includes the entered text and generation settings, so treat saved outputs as private.

**Not a translator:** choose the language that matches the text you entered. For best voice consistency, use a clean reference recording in that same language. The English/Hindi built-in voices are only convenient defaults; for other languages, upload a suitable reference recording.

**Text limit:** up to 12,000 characters per request. Long input is split into smaller pieces and joined with short pauses to avoid sending an oversized passage to a single model generation call. Listen to the joins for longer passages.

## 1. Optional Drive setup and install the pinned runtime

This uses the same Chatterbox upstream commit as the existing pronunciation notebook. Gradio is pinned to **6.31.0**. NumPy follows the pinned Chatterbox package metadata: Python 3.13 and newer use NumPy 2.x, while older Python versions use NumPy 1.x. The Colab runtime's Torch and torchaudio versions are preserved. Package installation progress is shown in the cell output.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import json
import hashlib
import gc
import time
import re
import threading
import uuid
import urllib.request
from datetime import datetime, timezone
from getpass import getpass

# Optional persistence. False means no Google Drive authorization is requested.
# Change to True before running this cell to preserve model files and generated outputs.
USE_GOOGLE_DRIVE = False

from google.colab import drive
if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Chatterbox Playground")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Google Drive persistence: ON")
else:
    DRIVE_ROOT = None
    print("Google Drive persistence: OFF (temporary Colab storage only)")

CHATTERBOX_COMMIT = "5de7a54aa4e5e2baadb0182dde554908b48b85c2"

# Match the NumPy requirement in the pinned Chatterbox pyproject.toml:
# Python 3.13+ requires NumPy 2.x; Python 3.12 and older use NumPy 1.x.
NUMPY_REQUIREMENT = (
    "numpy>=2.0.0" if sys.version_info >= (3, 13) else "numpy<2.0.0"
)
print("Python:", sys.version.split()[0], "| NumPy requirement:", NUMPY_REQUIREMENT)

# Let pip show download/build progress instead of appearing frozen.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        NUMPY_REQUIREMENT, "gradio==6.31.0",
    ],
    check=True,
)

import numpy as np
import torch
import torchaudio

TORCH_BEFORE = torch.__version__
TORCHAUDIO_BEFORE = torchaudio.__version__
print("Colab Python:", sys.version.split()[0])
print("Torch:", TORCH_BEFORE, "| torchaudio:", TORCHAUDIO_BEFORE)
print("NumPy:", np.__version__)

numpy_major = int(np.__version__.split(".")[0])
if sys.version_info >= (3, 13) and numpy_major < 2:
    raise RuntimeError("Python 3.13+ requires NumPy 2.x for this Chatterbox revision. Rerun the setup cell.")
if sys.version_info < (3, 13) and numpy_major >= 2:
    raise RuntimeError("Python 3.12 and older require NumPy 1.x for this Chatterbox revision. Rerun the setup cell.")

# Remove an incompatible PyPI installation if one is already present.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "chatterbox-tts"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

source_spec = f"git+https://github.com/resemble-ai/chatterbox.git@{CHATTERBOX_COMMIT}"
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        "--no-deps", source_spec,
    ],
    check=True,
)

dependency_specs = [
    "librosa==0.11.0",
    "s3tokenizer",
    "transformers==5.2.0",
    "diffusers==0.29.0",
    "git+https://github.com/resemble-ai/Perth.git@master",
    "conformer==0.3.2",
    "safetensors==0.5.3",
    "omegaconf",
    "soundfile",
    "huggingface_hub",
]
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--progress-bar", "on",
        *dependency_specs,
    ],
    check=True,
)

import librosa
import soundfile as sf
import gradio as gr
from safetensors import safe_open
from huggingface_hub import HfApi, snapshot_download
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

assert torch.__version__ == TORCH_BEFORE, (
    f"Torch changed during installation: {TORCH_BEFORE} -> {torch.__version__}"
)
assert torchaudio.__version__ == TORCHAUDIO_BEFORE, (
    f"torchaudio changed during installation: {TORCHAUDIO_BEFORE} -> {torchaudio.__version__}"
)

print("Gradio:", gr.__version__)
print("Pinned Chatterbox source:", CHATTERBOX_COMMIT)
print("Torch and torchaudio were preserved.")

## 2. Download or reuse the verified Multilingual V3 model files

The full model repository contains files that this notebook does not need. It downloads only the six files used by the existing project workflow, pins the download to a specific Hugging Face model revision, checks their sizes, and saves a verified copy in a dedicated Drive folder.

This model snapshot is separate from the audiobook project's existing model-snapshot folder.

In [ ]:
HF_HOME = Path("/content/chatterbox-playground-hf")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")

MODEL_REPO = "ResembleAI/chatterbox"
model_info = HfApi().model_info(MODEL_REPO, revision="main", files_metadata=True)
MODEL_REVISION = model_info.sha

MODEL_FILES = [
    "ve.pt",
    "s3gen.pt",
    "t3_mtl23ls_v3.safetensors",
    "grapheme_mtl_merged_expanded_v1.json",
    "conds.pt",
    "Cangjie5_TC.json",
]
REMOTE_SIZES = {
    item.rfilename: getattr(item, "size", None)
    for item in model_info.siblings
    if item.rfilename in MODEL_FILES
}
MINIMUM_SIZES = {
    "ve.pt": 5_000_000,
    "s3gen.pt": 1_000_000_000,
    "t3_mtl23ls_v3.safetensors": 2_000_000_000,
    "grapheme_mtl_merged_expanded_v1.json": 60_000,
    "conds.pt": 50_000,
    "Cangjie5_TC.json": 1_000_000,
}

DRIVE_MODEL_DIR = (
    DRIVE_ROOT / "model-snapshots" / f"chatterbox-v3-{MODEL_REVISION[:12]}"
    if USE_GOOGLE_DRIVE else None
)
LOCAL_MODEL_DIR = Path("/content/chatterbox-playground-model")


def check_snapshot(folder: Path) -> list[str]:
    problems = []
    for filename in MODEL_FILES:
        path = folder / filename
        if not path.is_file():
            problems.append(f"{filename}: missing")
            continue

        actual_size = path.stat().st_size
        expected_size = REMOTE_SIZES.get(filename)
        if expected_size and actual_size != expected_size:
            problems.append(
                f"{filename}: {actual_size:,} bytes; Hub reports {expected_size:,}"
            )
        elif not expected_size and actual_size < MINIMUM_SIZES[filename]:
            problems.append(f"{filename}: suspiciously small ({actual_size:,} bytes)")
    return problems


if LOCAL_MODEL_DIR.exists():
    shutil.rmtree(LOCAL_MODEL_DIR)
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE and not check_snapshot(DRIVE_MODEL_DIR):
    print("Reusing the verified Multilingual V3 snapshot from Drive.")
    for filename in MODEL_FILES:
        shutil.copy2(DRIVE_MODEL_DIR / filename, LOCAL_MODEL_DIR / filename)
else:
    if USE_GOOGLE_DRIVE and DRIVE_MODEL_DIR.exists():
        print("The dedicated Drive snapshot is incomplete or mismatched; replacing that snapshot only.")
        shutil.rmtree(DRIVE_MODEL_DIR)

    print("Downloading the selected V3 model files. This is several gigabytes...")
    snapshot_download(
        repo_id=MODEL_REPO,
        repo_type="model",
        revision=MODEL_REVISION,
        allow_patterns=MODEL_FILES,
        local_dir=str(LOCAL_MODEL_DIR),
        force_download=True,
    )

    problems = check_snapshot(LOCAL_MODEL_DIR)
    if problems:
        raise RuntimeError("Model snapshot verification failed:\n" + "\n".join(problems))

    if USE_GOOGLE_DRIVE:
        try:
            DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
            for filename in MODEL_FILES:
                shutil.copy2(LOCAL_MODEL_DIR / filename, DRIVE_MODEL_DIR / filename)

            (DRIVE_MODEL_DIR / "snapshot_manifest.json").write_text(
                json.dumps(
                    {
                        "repository": MODEL_REPO,
                        "revision": MODEL_REVISION,
                        "files": {
                            filename: {
                                "size_bytes": (DRIVE_MODEL_DIR / filename).stat().st_size
                            }
                            for filename in MODEL_FILES
                        },
                    },
                    indent=2,
                ),
                encoding="utf-8",
            )
            print("Verified model files saved to Drive.")
        except Exception as exc:
            print("Could not persist the model snapshot to Drive:", type(exc).__name__, str(exc))
            print("The verified local snapshot is still available for this runtime.")
    else:
        print("Model snapshot is in temporary Colab storage and will need downloading again after a runtime reset.")

problems = check_snapshot(LOCAL_MODEL_DIR)
if problems:
    raise RuntimeError("Local model snapshot is incomplete:\n" + "\n".join(problems))

print("Model revision:", MODEL_REVISION)
for filename in MODEL_FILES:
    print(f"{filename}: {(LOCAL_MODEL_DIR / filename).stat().st_size / 1024**2:.1f} MiB")

## 3. Preflight and load Chatterbox Multilingual V3

This checks the two PyTorch checkpoints with safe `weights_only=True` loading, checks the safetensors header, preserves the runtime's Torch versions, and loads the explicit V3 model selector. The notebook falls back to CPU if Colab did not provide a GPU.

In [ ]:
def safe_checkpoint_preflight(filename: str, force_refresh: bool = False) -> None:
    path = LOCAL_MODEL_DIR / filename
    try:
        state = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
            mmap=True,
        )
        if not isinstance(state, dict) or not state:
            raise RuntimeError(f"{filename} did not contain a non-empty state dictionary.")
        print(f"{filename}: safe checkpoint read ({len(state)} top-level entries)")
        del state
        gc.collect()
    except Exception as exc:
        if force_refresh:
            raise RuntimeError(
                f"{filename} still fails after a fresh Hub download: {type(exc).__name__}: {exc}"
            ) from exc
        print(f"{filename} failed preflight ({type(exc).__name__}: {exc}). Refreshing this file only.")
        snapshot_download(
            repo_id=MODEL_REPO,
            repo_type="model",
            revision=MODEL_REVISION,
            allow_patterns=[filename],
            local_dir=str(LOCAL_MODEL_DIR),
            force_download=True,
        )
        safe_checkpoint_preflight(filename, force_refresh=True)
        if USE_GOOGLE_DRIVE and DRIVE_MODEL_DIR.exists():
            shutil.copy2(path, DRIVE_MODEL_DIR / filename)


safe_checkpoint_preflight("ve.pt")
safe_checkpoint_preflight("s3gen.pt")

with safe_open(
    str(LOCAL_MODEL_DIR / "t3_mtl23ls_v3.safetensors"),
    framework="pt",
    device="cpu",
) as handle:
    if not list(handle.keys()):
        raise RuntimeError("The V3 T3 safetensors file is empty.")
    print("V3 T3 safetensors header: valid")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name, "| VRAM GiB:", round(gpu.total_memory / 1024**3, 1))
else:
    print("No GPU detected. Loading on CPU; generation may be very slow.")

print(f"Loading Chatterbox Multilingual V3 on {DEVICE}...")
load_started = time.perf_counter()
model = ChatterboxMultilingualTTS.from_local(
    str(LOCAL_MODEL_DIR),
    device=DEVICE,
    t3_model="v3",
)
load_seconds = time.perf_counter() - load_started

assert int(model.sr) == 24000, f"Expected 24 kHz output, got {model.sr}."
SUPPORTED_LANGUAGES = ChatterboxMultilingualTTS.get_supported_languages()
assert isinstance(SUPPORTED_LANGUAGES, dict) and "en" in SUPPORTED_LANGUAGES

LANGUAGE_CHOICES = [
    (f"{language_name} ({language_code})", language_code)
    for language_code, language_name in sorted(SUPPORTED_LANGUAGES.items())
]

print(f"Loaded in {load_seconds:.1f} seconds.")
print(f"Device: {DEVICE} | Output sample rate: {model.sr} Hz")
print("Supported language IDs:", ", ".join(sorted(SUPPORTED_LANGUAGES)))
print("The language dropdown in the next cell is populated from this model.")

## 4. Prepare reference voices, output folders, and synthesis helpers

A reference clip is used for voice conditioning, not as text to be transcribed. Use a clear sample with one speaker and little background noise. Around 8–10 seconds is a useful target; the helper accepts shorter clips down to one second.

The notebook includes the public Chatterbox English and Hindi demo references for quick testing. Your own recording can be uploaded or recorded from the microphone in the UI. The uploaded reference is normalized into a temporary Colab file and is not copied to Drive.

In [ ]:
REFERENCE_DIR = Path("/content/chatterbox-playground/references")
LOCAL_OUTPUT_DIR = Path("/content/chatterbox-playground/outputs")
DRIVE_OUTPUT_DIR = (DRIVE_ROOT / "generated") if USE_GOOGLE_DRIVE else None
for folder in (REFERENCE_DIR, LOCAL_OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if DRIVE_OUTPUT_DIR is not None:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEMO_REFERENCE_URLS = {
    "english": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac",
    "hindi": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/hi_f1.flac",
}
DEMO_REFERENCE_FILES = {
    language: REFERENCE_DIR / f"official_demo_{language}.flac"
    for language in DEMO_REFERENCE_URLS
}

for language, url in DEMO_REFERENCE_URLS.items():
    path = DEMO_REFERENCE_FILES[language]
    if not path.is_file() or path.stat().st_size < 1024:
        print(f"Downloading built-in {language} reference...")
        urllib.request.urlretrieve(url, path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def normalize_reference_audio(source_path: str | Path) -> tuple[Path, dict]:
    """Convert an uploaded/reference clip to mono 24 kHz PCM, first 10 s, 40 ms aligned."""
    source = Path(source_path)
    if not source.is_file():
        raise FileNotFoundError(f"Reference audio not found: {source}")

    audio, sample_rate = librosa.load(str(source), sr=24000, mono=True)
    if audio.ndim != 1 or audio.size == 0 or not np.isfinite(audio).all():
        raise ValueError("The reference audio is empty or invalid.")
    original_seconds = audio.size / 24000
    if original_seconds < 1.0:
        raise ValueError("Reference audio must contain at least one second of usable audio.")

    audio = audio[: 10 * 24000]
    pad = (-audio.size) % 960
    if pad:
        audio = np.pad(audio, (0, pad))

    digest = hashlib.sha256(audio.tobytes()).hexdigest()
    normalized_path = REFERENCE_DIR / f"normalized_{digest[:16]}.wav"
    if not normalized_path.is_file():
        sf.write(str(normalized_path), audio, 24000, subtype="PCM_16")

    return normalized_path, {
        "source_name": source.name,
        "source_sha256": sha256_file(source),
        "normalized_sha256": sha256_file(normalized_path),
        "original_duration_seconds": round(original_seconds, 3),
        "conditioning_duration_seconds": round(audio.size / 24000, 3),
        "sample_rate": 24000,
    }


def split_text_into_chunks(text: str, max_chars: int = 420) -> list[str]:
    """Pack sentences into conservative chunks; split very long sentences at word boundaries."""
    cleaned = text.strip()
    if not cleaned:
        raise ValueError("Enter some text first.")
    if len(cleaned) > MAX_TEXT_CHARACTERS:
        raise ValueError(f"Text is longer than the {MAX_TEXT_CHARACTERS:,}-character limit.")

    paragraphs = [part.strip() for part in cleaned.splitlines() if part.strip()]
    if not paragraphs:
        raise ValueError("Enter some text first.")

    chunks = []
    for paragraph in paragraphs:
        sentences = re.split(r"(?<=[.!?])\s+", paragraph)
        current = ""

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            if len(sentence) > max_chars:
                if current:
                    chunks.append(current)
                    current = ""
                words = sentence.split()
                piece = ""
                for word in words:
                    if len(word) > max_chars:
                        raise ValueError(
                            f"A single word/token is longer than the {max_chars}-character chunk limit."
                        )
                    candidate = word if not piece else piece + " " + word
                    if len(candidate) <= max_chars:
                        piece = candidate
                    else:
                        chunks.append(piece)
                        piece = word
                if piece:
                    chunks.append(piece)
                continue

            candidate = sentence if not current else current + " " + sentence
            if len(candidate) <= max_chars:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                current = sentence

        if current:
            chunks.append(current)

    if not chunks:
        raise ValueError("No usable text remained after splitting.")
    return chunks


MODEL_LOCK = threading.Lock()
INTER_CHUNK_PAUSE_SECONDS = 0.18
MAX_TEXT_CHARACTERS = 12000
MAX_CHUNK_CHARACTERS = 420

print("Reference voices cached.")
print("Local temporary output:", LOCAL_OUTPUT_DIR)
print("Drive output:", DRIVE_OUTPUT_DIR if DRIVE_OUTPUT_DIR is not None else "disabled")
print("Sentence-aware chunker ready.")

## 5. Build the text-to-speech function

Each request prepares the selected reference voice, generates the text chunks in order, inserts a short pause between chunks, writes a 24 kHz PCM WAV, and stores a metadata sidecar. A modest peak ceiling prevents accidental full-scale overshoots when the pieces are joined.

Only one generation request runs at a time because the model keeps voice-conditioning state in memory.

In [ ]:
def generate_speech(
    text: str,
    language_id: str,
    reference_mode: str,
    uploaded_reference: str | None,
    exaggeration: float,
    cfg_weight: float,
    temperature: float,
    repetition_penalty: float,
    min_p: float,
    top_p: float,
    seed_value: float,
    progress=gr.Progress(),
):
    started = time.perf_counter()

    try:
        chunks = split_text_into_chunks(text, max_chars=MAX_CHUNK_CHARACTERS)

        if language_id not in SUPPORTED_LANGUAGES:
            raise ValueError(f"Unsupported language ID: {language_id}")

        if reference_mode == "Upload/record my own voice":
            if not uploaded_reference:
                raise ValueError("Upload or record a reference audio clip first.")
            reference_source = Path(uploaded_reference)
            reference_label = "user-uploaded reference"
        elif reference_mode == "Built-in English demo voice":
            reference_source = DEMO_REFERENCE_FILES["english"]
            reference_label = "Chatterbox English demo"
        elif reference_mode == "Built-in Hindi demo voice":
            reference_source = DEMO_REFERENCE_FILES["hindi"]
            reference_label = "Chatterbox Hindi demo"
        elif reference_mode == "Automatic (English/Hindi demo; English fallback)":
            if language_id == "hi":
                reference_source = DEMO_REFERENCE_FILES["hindi"]
                reference_label = "automatic Hindi demo"
            else:
                reference_source = DEMO_REFERENCE_FILES["english"]
                reference_label = "automatic English demo"
        else:
            raise ValueError(f"Unknown reference mode: {reference_mode}")

        normalized_reference, reference_info = normalize_reference_audio(reference_source)

        request_seed = int(round(seed_value))
        if request_seed < 0:
            import secrets
            request_seed = secrets.randbelow(2**31 - 1)

        generation_parameters = {
            "exaggeration": float(exaggeration),
            "cfg_weight": float(cfg_weight),
            "temperature": float(temperature),
            "repetition_penalty": float(repetition_penalty),
            "min_p": float(min_p),
            "top_p": float(top_p),
        }

        progress(0, desc="Preparing reference voice...")
        pieces = []
        chunk_records = []

        with MODEL_LOCK:
            model.prepare_conditionals(
                str(normalized_reference),
                exaggeration=float(exaggeration),
            )

            for index, chunk in enumerate(chunks):
                chunk_seed = request_seed + index
                torch.manual_seed(chunk_seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed(chunk_seed)
                    torch.cuda.manual_seed_all(chunk_seed)

                progress(
                    (index + 1) / len(chunks),
                    desc=f"Generating chunk {index + 1}/{len(chunks)}",
                )
                generated = model.generate(
                    chunk,
                    language_id=language_id,
                    **generation_parameters,
                )
                waveform = generated.detach().float().cpu().numpy().squeeze()

                if waveform.ndim != 1 or waveform.size == 0 or not np.isfinite(waveform).all():
                    raise RuntimeError(f"Model returned an invalid waveform for chunk {index + 1}.")

                pieces.append(waveform.astype(np.float32, copy=False))
                chunk_records.append({
                    "chunk_index": index + 1,
                    "text": chunk,
                    "duration_seconds": round(waveform.size / int(model.sr), 3),
                    "seed": chunk_seed,
                })

                if index + 1 < len(chunks):
                    pieces.append(
                        np.zeros(
                            int(int(model.sr) * INTER_CHUNK_PAUSE_SECONDS),
                            dtype=np.float32,
                        )
                    )

        combined = np.concatenate(pieces).astype(np.float32, copy=False)
        peak_before_limiter = float(np.max(np.abs(combined)))
        limiter_gain = 1.0
        if peak_before_limiter > 0.98:
            limiter_gain = 0.98 / peak_before_limiter
            combined *= limiter_gain

        request_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
        local_wav = LOCAL_OUTPUT_DIR / f"chatterbox_{request_id}.wav"
        local_metadata = LOCAL_OUTPUT_DIR / f"chatterbox_{request_id}.json"

        sf.write(str(local_wav), combined, int(model.sr), format="WAV", subtype="PCM_16")

        metadata = {
            "request_id": request_id,
            "model": "ResembleAI/chatterbox Multilingual V3",
            "model_revision": MODEL_REVISION,
            "chatterbox_source_commit": CHATTERBOX_COMMIT,
            "device": DEVICE,
            "language_id": language_id,
            "language_name": SUPPORTED_LANGUAGES[language_id],
            "reference_label": reference_label,
            "reference": reference_info,
            "generation_parameters": generation_parameters,
            "seed": request_seed,
            "chunk_character_limit": MAX_CHUNK_CHARACTERS,
            "inter_chunk_pause_seconds": INTER_CHUNK_PAUSE_SECONDS,
            "chunks": chunk_records,
            "audio": {
                "sample_rate": int(model.sr),
                "duration_seconds": round(combined.size / int(model.sr), 3),
                "peak_before_limiter": round(peak_before_limiter, 6),
                "limiter_gain": round(limiter_gain, 6),
                "sha256": sha256_file(local_wav),
            },
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        local_metadata.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
        )

        # Drive persistence is optional. The original user reference recording
        # is never copied to Drive.
        if USE_GOOGLE_DRIVE and DRIVE_OUTPUT_DIR is not None:
            drive_wav = DRIVE_OUTPUT_DIR / local_wav.name
            drive_metadata = DRIVE_OUTPUT_DIR / local_metadata.name
            shutil.copy2(local_wav, drive_wav)
            shutil.copy2(local_metadata, drive_metadata)
            saved_location = f"Drive: {drive_wav}"
        else:
            saved_location = f"temporary Colab storage: {local_wav}"

        elapsed = time.perf_counter() - started
        status = (
            f"Generated {len(chunks)} chunk(s) in {elapsed:.1f}s on {DEVICE}. "
            f"Duration: {combined.size / int(model.sr):.1f}s. "
            f"Reference: {reference_label}. "
            f"Saved to {saved_location}. Download it before the runtime ends."
        )
        return str(local_wav), str(local_wav), status

    except Exception as exc:
        import traceback
        traceback.print_exc()
        return None, None, f"Generation failed: {type(exc).__name__}: {exc}"


print("Generation function ready.")
print("Maximum text:", MAX_TEXT_CHARACTERS, "characters.")
print("Chunk size:", MAX_CHUNK_CHARACTERS, "characters.")

## 6. Launch the Gradio web UI

The app uses a temporary Gradio share URL so it can be opened outside the Colab VM. Gradio's share tunnel is public-facing, so the notebook requires a username and password before launching. Choose a strong password and do not share the link or credentials.

The interface includes an audio upload/microphone control for your own voice reference. Generated WAV files are saved in the Google Drive folder shown in the UI status message.

In [ ]:
AUTH_USERNAME = input("Gradio login username [chatterbox]: ").strip() or "chatterbox"
AUTH_PASSWORD = getpass("Choose a strong Gradio password (minimum 12 characters): ")
if len(AUTH_PASSWORD) < 12:
    raise ValueError("Use a password of at least 12 characters.")

REFERENCE_MODE_CHOICES = [
    "Automatic (English/Hindi demo; English fallback)",
    "Built-in English demo voice",
    "Built-in Hindi demo voice",
    "Upload/record my own voice",
]

with gr.Blocks(
    title="Chatterbox Multilingual V3 TTS",
    analytics_enabled=False,
) as demo:
    gr.Markdown(
        """
        # Chatterbox Multilingual V3
        ### Custom text → speech

        Enter text, select its language, and choose a reference voice. This app generates speech,
        not translations. Longer passages are split into smaller chunks and joined in order.
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            text_input = gr.Textbox(
                label="Text to speak",
                placeholder="Type or paste your text here...",
                lines=12,
                max_lines=24,
            )
            language_input = gr.Dropdown(
                choices=LANGUAGE_CHOICES,
                value="en",
                label="Text language",
                interactive=True,
            )
            reference_mode_input = gr.Radio(
                choices=REFERENCE_MODE_CHOICES,
                value=REFERENCE_MODE_CHOICES[0],
                label="Voice reference",
            )
            reference_audio_input = gr.Audio(
                label="Upload or record a voice reference (used only in upload mode)",
                sources=["upload", "microphone"],
                type="filepath",
                format="wav",
            )
            gr.Markdown(
                "Tip: a clean 8–10 second sample of one speaker works well. "
                "For languages other than English or Hindi, upload a reference in the target language."
            )

        with gr.Column(scale=2):
            with gr.Accordion("Generation settings", open=True):
                exaggeration_input = gr.Slider(
                    minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                    label="Exaggeration",
                )
                cfg_input = gr.Slider(
                    minimum=0.0, maximum=1.0, value=0.5, step=0.05,
                    label="CFG weight",
                )
                temperature_input = gr.Slider(
                    minimum=0.05, maximum=2.0, value=0.7, step=0.05,
                    label="Temperature",
                )
                repetition_input = gr.Slider(
                    minimum=1.0, maximum=2.0, value=1.2, step=0.05,
                    label="Repetition penalty",
                )
                min_p_input = gr.Slider(
                    minimum=0.0, maximum=1.0, value=0.05, step=0.01,
                    label="Min-p",
                )
                top_p_input = gr.Slider(
                    minimum=0.0, maximum=1.0, value=0.95, step=0.01,
                    label="Top-p",
                )
                seed_input = gr.Number(
                    value=-1,
                    precision=0,
                    label="Seed (-1 = random)",
                )

            generate_button = gr.Button("Generate speech", variant="primary")

    audio_output = gr.Audio(label="Generated audio", type="filepath")
    file_output = gr.File(label="Download WAV")
    status_output = gr.Textbox(label="Status", interactive=False, lines=3)

    generate_button.click(
        fn=generate_speech,
        inputs=[
            text_input,
            language_input,
            reference_mode_input,
            reference_audio_input,
            exaggeration_input,
            cfg_input,
            temperature_input,
            repetition_input,
            min_p_input,
            top_p_input,
            seed_input,
        ],
        outputs=[audio_output, file_output, status_output],
        api_name=False,
    )

demo.queue()
print("Starting the password-protected Gradio app...")
print("After the next cell output appears, open the Gradio public URL.")
demo.launch(
    share=True,
    auth=(AUTH_USERNAME, AUTH_PASSWORD),
    server_name="0.0.0.0",
    debug=True,
    show_error=True,
    max_file_size="25mb",
    allowed_paths=[str(LOCAL_OUTPUT_DIR)],
    enable_monitoring=False,
)

## Usage notes and troubleshooting

- **Google Drive is optional:** the notebook does not request Drive access when `USE_GOOGLE_DRIVE = False`. This is the default. Generated WAVs and JSON metadata stay in the Colab VM and can be downloaded through Gradio, but are deleted when the runtime is recycled.
- **Enable Drive persistence:** set `USE_GOOGLE_DRIVE = True` in the setup cell before running it, then execute the notebook from the top. Authorize Google Drive when prompted. The V3 model snapshot and generated WAV/JSON files are then persisted under `MyDrive/Chatterbox Playground`.
- **Model download:** without Drive, the model is downloaded to the Colab VM each time you start a fresh runtime. With Drive enabled, the notebook reuses its verified model snapshot when available.
- **No GPU:** the notebook loads on CPU if CUDA is unavailable, but synthesis will be considerably slower.
- **Gradio link:** the share link is temporary and stops working when the Colab runtime disconnects. Relaunch the final cell to create a new one.
- **Password:** the Gradio login is separate from your Google account. Keep the URL and password private.
- **Voice reference:** use one speaker, clean speech, little background noise, and preferably 8–10 seconds in the same language as the text. A file is uploaded to the running Colab app for conditioning; the original upload and a normalized temporary copy remain in the active Colab runtime, but neither is copied to Drive. Terminate/reset the runtime to clear those temporary files.
- **Long passages:** the app splits text at sentence boundaries where possible and uses 420-character chunks. The short pauses between chunks are deliberate. Very long text can still take a while to synthesize.
- **Output privacy:** generated JSON metadata includes the entered text and generation settings. Keep the Drive folder or downloaded sidecars private.
- **Model mismatch:** if the V3 selector is missing or the notebook imports an old package, restart the Colab runtime and rerun from the top. The notebook intentionally uses the pinned upstream source commit rather than the old PyPI wheel.
- **This is not book synthesis:** nothing in this notebook changes the manuscript, pronunciation decisions, roadmap, or full-book synthesis lock.

Useful links: [Chatterbox source](https://github.com/resemble-ai/chatterbox) · [Gradio Audio component](https://gradio.app/docs/audio) · [Gradio Blocks](https://gradio.app/docs/gradio/blocks).